### Gold INEP — Indicadores Educacionais por UF

**Tabela:** `workspace.gold.fato_inep_educacao_uf_anual`

**Objetivo:** Disponibilizar indicadores anuais da educação superior por Unidade da Federação, área de formação CINE Brasil e modalidade de ensino, permitindo analisar a distribuição territorial da formação educacional e sua relação com a demanda por ocupações no mercado de trabalho formal brasileiro.

**Granularidade:** Ano × UF × área CINE detalhada × modalidade de ensino.

**Indicadores disponíveis:**
- Ingressantes, matrículas e concluintes dos cursos presenciais e EAD.
- Quantidade de cursos, vagas e inscritos exclusivamente da oferta presencial territorializada.

**Regras metodológicas:**
- `TP_DIMENSAO = 1`: indicadores de cursos presenciais.
- `TP_DIMENSAO = 2`: indicadores territorializados de estudantes EAD.
- `TP_DIMENSAO = 3`: excluído por representar informações EAD disponíveis apenas em nível nacional.
- `TP_DIMENSAO = 4`: excluído por representar oferta de instituições brasileiras no exterior.
- Registros sem UF não são atribuídos artificialmente a uma localização.
- Indicadores de cursos, vagas e inscritos EAD não são quantificados por UF; os campos de oferta presencial são nulos nas linhas EAD.

**Aplicações no projeto:**
- **EDA:** analisar diferenças regionais, evolução anual dos concluintes e distribuição das áreas de formação.
- **Machine Learning:** fornecer variáveis educacionais históricas para a futura Wide Table com o Novo Caged, após definir a correspondência CINE–CBO e a defasagem temporal adequada.
- **BI:** apoiar indicadores territoriais de formação e sua comparação com tendências de emprego.

**Limitação:** não representa a totalidade nacional dos indicadores de oferta EAD e não deve ser interpretada como uma base de formação mensal ou semestral.

### Gold INEP — Indicadores Educacionais Nacionais

**Tabela:** `workspace.gold.fato_inep_educacao_brasil_anual`

**Objetivo:** Consolidar indicadores anuais da educação superior brasileira em nível nacional, contemplando cursos presenciais e a distância, para avaliar tendências de oferta educacional, ingresso, matrícula e conclusão por área de formação.

**Granularidade:** Ano × área CINE detalhada × modalidade de ensino.

**Indicadores disponíveis:**
- Ingressantes, matrículas e concluintes.
- Quantidade de cursos, vagas e inscritos.
- Classificações CINE Brasil nos níveis geral, específico e detalhado.

**Regras metodológicas:**
- `TP_DIMENSAO = 1`: indicadores presenciais.
- `TP_DIMENSAO = 2`: ingressantes, matrículas e concluintes EAD, incluindo registros sem UF.
- `TP_DIMENSAO = 3`: quantidade de cursos, vagas e inscritos EAD em nível nacional.
- `TP_DIMENSAO = 4`: excluído por representar cursos ofertados por instituições brasileiras no exterior.
- As dimensões EAD são utilizadas de forma complementar, sem somar indiscriminadamente medidas de diferentes níveis geográficos.
- Valores excepcionais de vagas e inscritos da dimensão 2 não são incorporados automaticamente à oferta nacional.

**Aplicações no projeto:**
- **EDA:** comparar a evolução da educação presencial e EAD, identificar áreas de formação em crescimento e analisar ingressantes, matrículas e concluintes.
- **Machine Learning:** fornecer indicadores nacionais de contexto e tendências educacionais, quando pertinentes ao modelo.
- **BI:** apresentar uma visão consolidada da educação superior brasileira, incluindo indicadores que não podem ser distribuídos por UF.

**Limitação:** a tabela não possui granularidade estadual ou municipal e, portanto, não deve ser diretamente associada às UFs do Novo Caged como se os indicadores nacionais fossem valores regionais.

![image_1793236104097.png](./image_1793236104097.png "image_1793236104097.png")

In [0]:
from pyspark.sql import functions as F

TABELA_SILVER = "workspace.silver.inep_superior_cursos"
TABELA_GOLD_UF = "workspace.gold.fato_inep_educacao_uf_anual"
TABELA_GOLD_BRASIL = "workspace.gold.fato_inep_educacao_brasil_anual"

ANOS_PROCESSAMENTO = [2021, 2022, 2023, 2024]

VERSAO_GOLD_UF = "inep_gold_uf_v2"
VERSAO_GOLD_BRASIL = "inep_gold_brasil_v2"

df_silver = spark.table(TABELA_SILVER).filter(F.col("nome_uf").isNotNull())

In [0]:
# 02 - Identificar períodos pendentes

# Controle da origem Silver
linhas_silver = (
    df_silver
    .filter(
        (F.col("ano_censo").isin(ANOS_PROCESSAMENTO))
        & (F.col("nome_uf").isNotNull())
    )
    .groupBy("ano_censo")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("hash_origem").alias("qtd_hashes"),
        F.first("hash_origem").alias("hash"),
        F.countDistinct("versao_transformacao").alias("qtd_versoes"),
        F.first("versao_transformacao").alias("versao")
    )
    .collect()
)

controle_silver = {}

for r in linhas_silver:
    ano = int(r["ano_censo"])

    if (
        r["registros"] <= 0
        or r["qtd_hashes"] != 1
        or r["hash"] is None
        or r["qtd_versoes"] != 1
        or r["versao"] is None
    ):
        raise ValueError(f"Silver inconsistente: {ano}")

    controle_silver[ano] = r.asDict()

if set(controle_silver) != set(ANOS_PROCESSAMENTO):
    raise ValueError("Existem anos ausentes na Silver.")


def identificar_pendencias(tabela, versao_esperada):

    if not spark.catalog.tableExists(tabela):
        return ANOS_PROCESSAMENTO.copy()

    df_gold_atual = spark.table(tabela)

    campos_controle = [
        "hash_silver_origem",
        "versao_silver_origem",
        "registros_silver_origem",
        "versao_transformacao"
    ]

    # Tabela criada na primeira versão, sem metadados novos
    if any(c not in df_gold_atual.columns for c in campos_controle):
        return ANOS_PROCESSAMENTO.copy()

    linhas_gold = (
        df_gold_atual
        .filter(
            (F.col("ano_censo").isin(ANOS_PROCESSAMENTO))
        )
        .groupBy("ano_censo")
        .agg(
            F.count("*").alias("registros_gold"),
            *[
                F.countDistinct(c).alias(f"qtd_{c}")
                for c in campos_controle
            ],
            *[
                F.first(c).alias(c)
                for c in campos_controle
            ]
        )
        .collect()
    )

    controle_gold = {
        int(r["ano_censo"]): r.asDict()
        for r in linhas_gold
    }

    pendentes = []

    for ano in ANOS_PROCESSAMENTO:
        origem = controle_silver[ano]
        atual = controle_gold.get(ano)

        atualizado = (
            atual is not None
            and atual["registros_gold"] > 0
            and all(
                atual[f"qtd_{c}"] == 1
                for c in campos_controle
            )
            and atual["hash_silver_origem"] == origem["hash"]
            and atual["versao_silver_origem"] == origem["versao"]
            and atual["registros_silver_origem"] == origem["registros"]
            and atual["versao_transformacao"] == versao_esperada
        )

        if not atualizado:
            pendentes.append(ano)

    return pendentes


anos_pendentes_uf = identificar_pendencias(
    TABELA_GOLD_UF, VERSAO_GOLD_UF
)

anos_pendentes_brasil = identificar_pendencias(
    TABELA_GOLD_BRASIL, VERSAO_GOLD_BRASIL
)

anos_pendentes = sorted(
    set(anos_pendentes_uf) | set(anos_pendentes_brasil)
)

print("Gold UF pendente:", anos_pendentes_uf)
print("Gold Brasil pendente:", anos_pendentes_brasil)

if not anos_pendentes:
    dbutils.notebook.exit("SEM_PENDENCIAS")

In [0]:
df_inep = df_silver.filter(
    F.col("ano_censo").isin(anos_pendentes)
)

### Construção da Agregação Gold por UF 
Gold educacional destinada ao EDA e à futura integração com o Caged
**As regras serão:**
- Ingressantes, matrículas e concluintes: dimensões 1 e 2.
- Vagas, inscritos e quantidade de cursos: somente dimensão 1, identificados explicitamente como oferta presencial.
- Registros sem UF: não entram na fato territorial, mas continuam disponíveis na Silver e serão considerados na futura Gold nacional.
- Dimensões 3 e 4: não entram na fato por UF.

In [0]:
from pyspark.sql import functions as F

# Registros com localização territorial válida
df_uf = (
    df_inep
    .filter(F.col("tipo_dimensao").isin(1, 2))
    .filter(
        F.col("sigla_uf").rlike("^[A-Z]{2}$")
    )
)

# Selecionar somente as dimensões necessárias
df_gold_uf = (
    df_uf
    .groupBy(
        "ano_censo",
        "sigla_uf",
        "codigo_cine_rotulo",
        "nome_cine_rotulo",
        "codigo_cine_area_geral",
        "nome_cine_area_geral",
        "codigo_cine_area_especifica",
        "nome_cine_area_especifica",
        "codigo_cine_area_detalhada",
        "nome_cine_area_detalhada",
        "tipo_modalidade_ensino"
    )
    .agg(
        # Indicadores de estudantes: presencial + EAD
        F.sum("quantidade_ingressantes").alias("ingressantes"),
        F.sum("quantidade_matriculas").alias("matriculas"),
        F.sum("quantidade_concluintes").alias("concluintes"),

        # Indicadores de oferta: somente presencial
        F.sum(
            F.when(
                F.col("tipo_dimensao") == 1,
                F.col("quantidade_cursos")
            ).otherwise(0)
        ).alias("cursos_presenciais"),

        F.sum(
            F.when(
                F.col("tipo_dimensao") == 1,
                F.col("quantidade_vagas")
            ).otherwise(0)
        ).alias("vagas_presenciais"),

        F.sum(
            F.when(
                F.col("tipo_dimensao") == 1,
                F.col("quantidade_inscritos")
            ).otherwise(0)
        ).alias("inscritos_presenciais")
    )
)

# Oferta presencial não se aplica à modalidade EAD por UF
for campo in [
    "cursos_presenciais",
    "vagas_presenciais",
    "inscritos_presenciais"
]:
    df_gold_uf = df_gold_uf.withColumn(
        campo,
        F.when(
            F.col("tipo_modalidade_ensino") == 1,
            F.col(campo)
        )
    )

### Gold Inep Nacional
**Regras:**
- Estudantes: dimensões 1 e 2.
- Cursos, vagas e inscritos: dimensões 1 e 3.
- Exterior: fora da agregação nacional brasileira.

In [0]:
from pyspark.sql import functions as F

# Dimensões utilizadas na visão Brasil
df_brasil = df_inep.filter(
    F.col("tipo_dimensao").isin(1, 2, 3)
)

# Preservamos a hierarquia CINE e a modalidade
dimensoes = [
    "ano_censo",
    "codigo_cine_rotulo",
    "nome_cine_rotulo",
    "codigo_cine_area_geral",
    "nome_cine_area_geral",
    "codigo_cine_area_especifica",
    "nome_cine_area_especifica",
    "codigo_cine_area_detalhada",
    "nome_cine_area_detalhada",
    "tipo_modalidade_ensino"
]

df_gold_brasil = (
    df_brasil
    .groupBy(*dimensoes)
    .agg(
        # Estudantes: presencial (1) e EAD territorializado (2)
        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(1, 2),
                F.col("quantidade_ingressantes")
            ).otherwise(0)
        ).alias("ingressantes"),

        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(1, 2),
                F.col("quantidade_matriculas")
            ).otherwise(0)
        ).alias("matriculas"),

        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(1, 2),
                F.col("quantidade_concluintes")
            ).otherwise(0)
        ).alias("concluintes"),

        # Oferta: presencial (1) e EAD nacional (3)
        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(1, 3),
                F.col("quantidade_cursos")
            ).otherwise(0)
        ).alias("cursos"),

        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(1, 3),
                F.col("quantidade_vagas")
            ).otherwise(0)
        ).alias("vagas"),

        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(1, 3),
                F.col("quantidade_inscritos")
            ).otherwise(0)
        ).alias("inscritos")
    )
)

print("DataFrame nacional preparado.")

display(
    df_gold_brasil
    .orderBy(
        "ano_censo",
        "codigo_cine_area_detalhada",
        "tipo_modalidade_ensino"
    )
    .limit(20)
)

### Validações Gerais

In [0]:
metricas = [
    ("quantidade_ingressantes", "ingressantes"),
    ("quantidade_matriculas", "matriculas"),
    ("quantidade_concluintes", "concluintes")
]

origem = (
    df_uf.groupBy("ano_censo")
    .agg(*[
        F.sum(campo_origem).alias(campo_destino)
        for campo_origem, campo_destino in metricas
    ])
)

destino = (
    df_gold_uf.groupBy("ano_censo")
    .agg(*[
        F.sum(campo_destino).alias(campo_destino)
        for _, campo_destino in metricas
    ])
)

comparacao = origem.alias("b").join(
    destino.alias("g"),
    "ano_censo"
)

for _, campo in metricas:
    comparacao = comparacao.withColumn(
        f"diferenca_{campo}",
        F.col(f"b.{campo}") - F.col(f"g.{campo}")
    )

display(
    comparacao.select(
        "ano_censo",
        *[f"diferenca_{campo}" for _, campo in metricas]
    ).orderBy("ano_censo")
)

In [0]:
from pyspark.sql import functions as F

chaves_gold = [
    "ano_censo",
    "sigla_uf",
    "codigo_cine_rotulo",
    "tipo_modalidade_ensino"
]

# Verificar duplicidades
df_duplicados = (
    df_gold_uf
    .groupBy(*chaves_gold)
    .count()
    .filter(F.col("count") > 1)
)

qtd_duplicados = df_duplicados.count()

print(f"Chaves duplicadas: {qtd_duplicados}")

if qtd_duplicados > 0:
    display(df_duplicados)
    raise ValueError(
        "Existem duplicidades na chave analítica da Gold."
    )

# Validar preenchimento da chave
df_chaves_nulas = df_gold_uf.filter(
    F.col("ano_censo").isNull()
    | F.col("sigla_uf").isNull()
    | F.col("codigo_cine_area_detalhada").isNull()
    | F.col("tipo_modalidade_ensino").isNull()
)

qtd_chaves_nulas = df_chaves_nulas.count()

print(f"Registros com chaves nulas: {qtd_chaves_nulas}")

if qtd_chaves_nulas > 0:
    display(df_chaves_nulas.limit(20))
    raise ValueError(
        "Existem chaves nulas na Gold."
    )

print("Chave analítica validada!")

In [0]:
metricas = {
    "ingressantes": (
        "quantidade_ingressantes", [1, 2]
    ),
    "matriculas": (
        "quantidade_matriculas", [1, 2]
    ),
    "concluintes": (
        "quantidade_concluintes", [1, 2]
    ),
    "cursos": (
        "quantidade_cursos", [1, 3]
    ),
    "vagas": (
        "quantidade_vagas", [1, 3]
    ),
    "inscritos": (
        "quantidade_inscritos", [1, 3]
    )
}

# Totais esperados calculados na Silver
expressoes = []

for destino, (origem, dimensoes_validas) in metricas.items():
    expressoes.append(
        F.sum(
            F.when(
                F.col("tipo_dimensao").isin(dimensoes_validas),
                F.col(origem)
            ).otherwise(0)
        ).alias(destino)
    )

df_totais_silver = (
    df_brasil
    .groupBy("ano_censo")
    .agg(*expressoes)
)

# Totais produzidos na Gold nacional
df_totais_gold = (
    df_gold_brasil
    .groupBy("ano_censo")
    .agg(
        *[
            F.sum(nome).alias(nome)
            for nome in metricas
        ]
    )
)

# Comparar as seis medidas
df_comparacao = df_totais_silver.alias("s").join(
    df_totais_gold.alias("g"),
    "ano_censo"
)

df_diferencas = df_comparacao.select(
    "ano_censo",
    *[
        (
            F.col(f"s.{nome}") -
            F.col(f"g.{nome}")
        ).alias(f"diferenca_{nome}")
        for nome in metricas
    ]
).orderBy("ano_censo")

display(df_diferencas)

In [0]:
chaves_brasil = [
    "ano_censo",
    "codigo_cine_rotulo",
    "tipo_modalidade_ensino"
]

# Verificar duplicidades
duplicados = (
    df_gold_brasil
    .groupBy(*chaves_brasil)
    .count()
    .filter(F.col("count") > 1)
)

qtd_duplicados = duplicados.count()

print("Chaves duplicadas:", qtd_duplicados)

if qtd_duplicados > 0:
    display(duplicados.limit(20))

# Verificar chaves nulas
condicao_nulos = (
    F.col("ano_censo").isNull()
    | F.col("codigo_cine_area_detalhada").isNull()
    | F.col("tipo_modalidade_ensino").isNull()
)

qtd_nulos = df_gold_brasil.filter(
    condicao_nulos
).count()

print("Chaves nulas:", qtd_nulos)

if qtd_duplicados > 0 or qtd_nulos > 0:
    raise ValueError(
        "A Gold nacional apresenta inconsistências nas chaves."
    )

print("Chaves da Gold nacional validadas.")

In [0]:
colunas = [
    "diferenca_ingressantes",
    "diferenca_matriculas",
    "diferenca_concluintes"
]

condicao_erro = F.lit(False)

for c in colunas:
    condicao_erro = (
        condicao_erro
        | F.col(c).isNull()
        | (F.col(c) != 0)
    )

df_diferencas_uf = comparacao.select(
    "ano_censo",
    *[F.col(f"diferenca_{c}") for c in [
        "ingressantes",
        "matriculas",
        "concluintes"
    ]]
)

if df_diferencas_uf.filter(condicao_erro).limit(1).count():
    raise ValueError("Divergência nos indicadores Gold UF.")

### Gravação dos Dados Analiticos

In [0]:
# 10 - Gravar Gold UF

# Função para gravar a Gold incrementalmente
def gravar_gold_incremental(df, tabela, anos, versao):

    for ano in anos:

        origem = controle_silver[ano]

        df_ano = (
            df.filter(F.col("ano_censo") == ano)
            .withColumn(
                "hash_silver_origem",
                F.lit(origem["hash"])
            )
            .withColumn(
                "versao_silver_origem",
                F.lit(origem["versao"])
            )
            .withColumn(
                "registros_silver_origem",
                F.lit(origem["registros"])
            )
            .withColumn(
                "versao_transformacao",
                F.lit(versao)
            )
            .withColumn(
                "dt_processamento",
                F.current_timestamp()
            )
        )

        if df_ano.limit(1).count() == 0:
            raise ValueError(
                f"{tabela}: sem registros para {ano}"
            )

        if not spark.catalog.tableExists(tabela):

            (
                df_ano.write
                .format("delta")
                .mode("error")
                .saveAsTable(tabela)
            )

        else:

            (
                df_ano.write
                .format("delta")
                .mode("overwrite")
                .option(
                    "replaceWhere",
                    f"ano_censo = {ano}"
                )
                .option("mergeSchema", "true")
                .saveAsTable(tabela)
            )

        print(f"{tabela}: {ano} atualizado.")


gravar_gold_incremental(
    df_gold_uf,
    TABELA_GOLD_UF,
    anos_pendentes_uf,
    VERSAO_GOLD_UF
)

In [0]:
# 11 - Gravar Gold Brasil

gravar_gold_incremental(
    df_gold_brasil,
    TABELA_GOLD_BRASIL,
    anos_pendentes_brasil,
    VERSAO_GOLD_BRASIL
)

In [0]:
# 12 - Validação final

pendencias_uf = identificar_pendencias(
    TABELA_GOLD_UF,
    VERSAO_GOLD_UF
)

pendencias_brasil = identificar_pendencias(
    TABELA_GOLD_BRASIL,
    VERSAO_GOLD_BRASIL
)

if pendencias_uf or pendencias_brasil:
    raise ValueError(
        f"Pendências Gold UF: {pendencias_uf}; "
        f"Gold Brasil: {pendencias_brasil}"
    )

print("Gold UF e Gold Brasil atualizadas.")
print("Controle incremental validado.")

dbutils.notebook.exit("GOLD_ATUALIZADA")

### Analise Exploratoria dos Dados

In [0]:
%skip
df_analise_dimensao = (
    df_inep
    .groupBy(
        "ano_censo",
        "tipo_dimensao",
        "tipo_modalidade_ensino"
    )
    .agg(
        F.count("*").alias("registros"),

        F.sum(
            F.when(
                F.col("sigla_uf").isNull()
                | (F.trim(F.col("sigla_uf")) == ""),
                1
            ).otherwise(0)
        ).alias("registros_sem_uf"),

        F.countDistinct("sigla_uf").alias("ufs_distintas"),

        F.sum("quantidade_cursos").alias("cursos"),
        F.sum("quantidade_vagas").alias("vagas"),
        F.sum("quantidade_inscritos").alias("inscritos"),
        F.sum("quantidade_ingressantes").alias("ingressantes"),
        F.sum("quantidade_matriculas").alias("matriculas"),
        F.sum("quantidade_concluintes").alias("concluintes")
    )
    .orderBy(
        "ano_censo",
        "tipo_dimensao",
        "tipo_modalidade_ensino"
    )
)

display(df_analise_dimensao)

In [0]:
%skip
df_analise_ead = (
    df_inep
    .filter(F.col("tipo_dimensao").isin(2, 3))
    .groupBy("ano_censo", "tipo_dimensao")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("codigo_curso").alias("cursos_distintos"),
        F.countDistinct("codigo_ies").alias("ies_distintas"),
        F.sum("quantidade_cursos").alias("quantidade_cursos"),
        F.sum("quantidade_vagas").alias("quantidade_vagas"),
        F.sum("quantidade_ingressantes").alias("ingressantes"),
        F.sum("quantidade_matriculas").alias("matriculas"),
        F.sum("quantidade_concluintes").alias("concluintes")
    )
    .orderBy("ano_censo", "tipo_dimensao")
)

display(df_analise_ead)

In [0]:
%skip
from pyspark.sql import functions as F

df_excecoes = (
    df_inep
    .filter(F.col("tipo_dimensao") == 2)
    .filter(
        F.col("sigla_uf").isNull()
        | (F.trim(F.col("sigla_uf")) == "")
        | (F.coalesce(F.col("quantidade_vagas"), F.lit(0)) > 0)
        | (F.coalesce(F.col("quantidade_inscritos"), F.lit(0)) > 0)
    )
)

display(
    df_excecoes.select(
        "ano_censo",
        "sigla_uf",
        "codigo_ies",
        "codigo_curso",
        "nome_curso",
        "codigo_cine_rotulo",
        "tipo_dimensao",
        "quantidade_vagas",
        "quantidade_inscritos",
        "quantidade_ingressantes",
        "quantidade_matriculas",
        "quantidade_concluintes"
    ).orderBy("ano_censo", "sigla_uf")
)

display(
    df_excecoes
    .groupBy("ano_censo")
    .agg(
        F.count("*").alias("registros_excecao"),
        F.sum(
            F.when(
                F.col("sigla_uf").isNull()
                | (F.trim(F.col("sigla_uf")) == ""),
                1
            ).otherwise(0)
        ).alias("registros_sem_uf"),
        F.sum("quantidade_vagas").alias("vagas"),
        F.sum("quantidade_inscritos").alias("inscritos"),
        F.sum("quantidade_ingressantes").alias("ingressantes"),
        F.sum("quantidade_matriculas").alias("matriculas"),
        F.sum("quantidade_concluintes").alias("concluintes")
    )
    .orderBy("ano_censo")
)